# ChpWiseDB — Policy Compliance Pipeline

| Component | Model |
|-----------|-------|
| Embedding | BGE-large `BAAI/bge-large-en-v1.5` |
| Reranker  | Qwen3-Reranker-4B |
| LLM       | Mistral-7B-Instruct-v0.3 *(Section 9 — run after validation)* |

**`qwen_norm_score`**: yes/no log-softmax probability → absolute 0–1 relevance score

---

## Run Order
Run **Sections 1–8** first. Only run **Section 9** once Section 8 retrieval accuracy is satisfactory.

| Section | Purpose | Run now? |
|---------|---------|----------|
| 1 | Imports & Config | ✅ |
| 2 | Chunk & index hospital policy | ✅ |
| 3 | Chunk & index JCI ESS | ✅ |
| 4 | Load BGE-large + retrieval | ✅ |
| 5 | Load Qwen reranker | ✅ |
| 6 | Query parser | ✅ |
| 7 | Gold standard queries | ✅ |
| 8 | Validation loop (no LLM) | ✅ |
| 9 | Compliance analysis (Mistral) | ⏸ after Section 8 |
| 10 | Summary | ✅ after Section 8 |

---
## Section 1 — Imports & Config
All paths, model IDs, retrieval settings, and parser keyword lists are declared here.
Update `JCI_PDF` and `POLICY_PDF` to match your local file paths.

In [1]:
import os
import re
import time
import json
import numpy as np
import pandas as pd
import torch
import transformers

from collections import defaultdict, Counter
from dataclasses import dataclass, asdict

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from transformers import AutoTokenizer, AutoModelForCausalLM

# ── Paths ────────────────────────────────────────────────────
JCI_PDF        = 'ESS Standards_7th Ed .pdf'
POLICY_PDF     = 'old Transfer 290523 .pdf'
DB_SAVE_DIR    = 'saved_dbs'
POLICY_DB_NAME = 'TransferPolicy'

# ── Embedding model ──────────────────────────────────────────
EMBEDDING_MODEL_NAME   = 'BGE-large'
EMBEDDING_MODEL_STRING = 'BAAI/bge-large-en-v1.5'

# ── Reranker ─────────────────────────────────────────────────
QWEN_MODEL_ID = 'Qwen/Qwen3-Reranker-4B'

# ── LLM (used in Section 9 only) ────────────────────────────
LLM_MODEL_ID = 'mistralai/Mistral-7B-Instruct-v0.3'

# ── Retrieval settings ───────────────────────────────────────
PER_DB_K = 8    # chunks fetched per chapter DB
FINAL_K  = 10   # chunks kept after global re-sort and dedup

# ── LLM settings (Section 9 only) ───────────────────────────
LLM_MAX_TOKENS = 512

# ── JCI chapter codes ────────────────────────────────────────
VALID_CODES = {
    'APR', 'IPSG', 'ACC', 'PCC', 'AOP', 'COP', 'ASC', 'MMU',
    'QPS', 'PCI',  'GLD', 'FMS', 'SQE', 'MOI', 'MPE', 'HRP',
}
END_OF_BOOK_MARKERS = ['glossary', 'index', 'appendix', 'references']

# ── Parser config ────────────────────────────────────────────
VALID_CHAPTERS = VALID_CODES   # same set, aliased for parser use

KNOWN_POLICIES = {
    'transfer':          'Transfer Policy',
    'discharge':         'Discharge Policy',
    'admission':         'Admission Policy',
    'icu':               'ICU Policy',
    'infection control': 'Infection Control Policy',
}

TASK_KEYWORDS = {
    'compliance_check': [
        'comply', 'complies', 'compliance', 'aligned', 'aligns',
        'meets', 'meet', 'satisfies', 'consistent with', 'in line with',
        'conform', 'gap', 'gaps', 'missing', 'is this', 'does this',
        'adequate', 'fulfil',
    ],
    'comparison': [
        'compare', 'comparison', 'versus', 'vs', 'differ',
        'difference', 'how does', 'contrast', 'against',
    ],
    'clarification': [
        'what does', 'what is', 'what are', 'explain', 'define',
        'describe', 'clarify', 'what requirement', 'what criteria',
        'what should', 'how should',
    ],
}

print('Section 1 complete — config loaded.')

/Users/siri/Desktop/sghpolicyreview/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Section 1 complete — config loaded.


---
## Section 2 — Load, Chunk & Build FAISS DB: Hospital Policy

- Loads the hospital policy PDF and splits into overlapping chunks
- Detects clause numbers (e.g. `5.1.1`) and attaches them as metadata labels
- Builds and saves a FAISS vector DB using BGE-large embeddings
- **Run once** — skips if DB already exists

> **Sanity check**: inspect the 3 sample chunks printed below before proceeding.

In [2]:
def load_and_chunk_policy(path: str, chunk_size: int = 1000,
                           chunk_overlap: int = 250) -> list:
    """
    Loads the hospital policy PDF and splits into chunks.
    Attaches clause number as metadata label where detectable.
    """
    pages     = PyPDFLoader(path).load()
    full_text = '\n'.join(p.page_content for p in pages)

    # Basic cleaning
    full_text = re.sub(r'Page\s+\d+\s+of\s+\d+', '', full_text)
    full_text = re.sub(r'\n{3,}', '\n\n', full_text)

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=['\n\n', '\n', '. ', ' '],
    )

    docs = []
    for i, chunk in enumerate(splitter.split_text(full_text)):
        if len(chunk.strip()) < 100:
            continue
        clause_match = re.search(r'\b(\d+\.\d+(?:\.\d+)?)\b', chunk)
        clause_label = clause_match.group(1) if clause_match else 'UNKNOWN'
        docs.append(Document(
            page_content=chunk.strip(),
            metadata={
                'source':        POLICY_DB_NAME,
                'clause_number': clause_label,
                'chunk_index':   i,
            }
        ))

    print(f'Policy: {len(docs)} chunks created.')
    return docs


os.makedirs(DB_SAVE_DIR, exist_ok=True)
policy_db_path = os.path.join(DB_SAVE_DIR, EMBEDDING_MODEL_NAME, POLICY_DB_NAME)

policy_chunks = load_and_chunk_policy(POLICY_PDF)

# ── Sanity check — inspect before building DB ────────────────
print('\nSample policy chunks:')
for doc in policy_chunks[:3]:
    print(f"  clause : {doc.metadata['clause_number']}")
    print(f"  text   : {doc.page_content[:200]}")
    print()

# ── Build and save FAISS DB ──────────────────────────────────
if os.path.exists(policy_db_path):
    print(f'Policy DB already exists — skipping build.')
else:
    print('Building policy FAISS DB...')
    policy_embeddings = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL_STRING,
        encode_kwargs={'normalize_embeddings': True},
    )
    policy_db = FAISS.from_documents(policy_chunks, policy_embeddings)
    policy_db.save_local(policy_db_path)
    print(f'Policy DB saved → {policy_db_path}')

print('Section 2 complete.')

Policy: 18 chunks created.

Sample policy chunks:
  clause : UNKNOWN
  text   : Singapore General Hospital
Policy & Procedure
Title:
Intra-Hospital and Inter-Hospital Transfer
Document No:
MB-ACC-004

  clause : 1.1
  text   : Issuing Department: OPSQ-Clinical Governance and
Quality (CGQ)
Issued By: Dr Chee Huei Leng, Director of OPSQ-CGQ
Date Revised: XX Jun 2023
Version Number: 7
Approved By:
A/Prof Ruban Poopalalingam,
C

  clause : 3.1
  text   : transfer should only be made when the benefits outweigh the risks.
3. STANDARD
3.1 The reason for transfer should be the need for additional care that is not available at
the patient’s current locatio

Building policy FAISS DB...


/tmp/ipykernel_384/1737233924.py:56: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  policy_embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3843.23it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Policy DB saved → saved_dbs/BGE-large/TransferPolicy
Section 2 complete.


---
## Section 3 — Load, Chunk & Build FAISS DB: JCI ESS

- Loads the JCI ESS PDF and groups pages by chapter
- Splits each chapter into standard-level chunks with `standard_code` metadata
- Builds one FAISS DB per chapter under `saved_dbs/BGE-large/<chapter>/`
- **Run once** — skips if DBs already exist

In [3]:
def load_pdf(path: str):
    pages = PyPDFLoader(path).load()
    print(f'Loaded {len(pages)} pages from JCI PDF.')
    return pages


def detect_chapter(text: str):
    """Returns JCI chapter code if this page is a chapter title page."""
    for line in text.split('\n'):
        m = re.match(r'^.+\(([A-Z]{3,4})\)\s*$', line.strip())
        if m and m.group(1) in VALID_CODES:
            return m.group(1)
    return None


def is_end_of_book(text: str) -> bool:
    """True if page looks like back-matter."""
    return any(marker in text.strip()[:200].lower()
               for marker in END_OF_BOOK_MARKERS)


def group_pages_by_chapter(pages) -> dict:
    """Assigns each PDF page to its chapter based on detected headings."""
    chapter_pages = defaultdict(list)
    current = 'UNKNOWN'
    for page in pages:
        detected = detect_chapter(page.page_content)
        if detected:
            current = detected
        chapter_pages[current].append(page)

    if 'HRP' in chapter_pages:
        before = len(chapter_pages['HRP'])
        chapter_pages['HRP'] = [
            p for p in chapter_pages['HRP']
            if not is_end_of_book(p.page_content)
        ]
        print(f'HRP: removed {before - len(chapter_pages["HRP"])} end-of-book pages.')

    print('\n=== Chapter page counts ===')
    for ch, pgs in chapter_pages.items():
        print(f'  {ch}: {len(pgs)} pages')
    return chapter_pages


def build_chapter_chunks(chapter_pages: dict, chunk_size: int = 1000,
                          chunk_overlap: int = 250) -> dict:
    """Splits each chapter into chunks, one per standard where possible."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=['\n\n', '\n', '. ', ' '],
    )
    chapter_chunks = {}

    for chapter, pg_list in chapter_pages.items():
        full_text = '\n'.join(p.page_content for p in pg_list)
        full_text = re.sub(r'Page\s+\d+\s+of\s+\d+', '', full_text)
        full_text = re.sub(r'\n{3,}', '\n\n', full_text)

        toc = re.search(
            r'Standards,\s+Intents,\s+and\s+Measurable\s+Elements',
            full_text, re.IGNORECASE
        )
        if toc:
            full_text = full_text[toc.start():]

        pattern = (
            rf'(?:\n|^)(?:Standard|Requirement)\s+'
            rf'({chapter}\s*\.\s*\d+(?:\s*\.\s*\d+)?(?:\s*\.\s*\d+)?)\b'
        )
        splits = re.split(pattern, full_text)

        docs = []
        if splits[0].strip():
            docs.append(Document(
                page_content=splits[0].strip(),
                metadata={'chapter': chapter, 'standard_code': 'INTRO'},
            ))

        for i in range(1, len(splits), 2):
            if i + 1 >= len(splits):
                break
            code    = splits[i].strip().replace(' ', '')
            content = f'{code}\n{splits[i + 1]}'

            if len(content.strip()) < 100:
                continue
            if len(content) <= chunk_size:
                docs.append(Document(
                    page_content=content.strip(),
                    metadata={'chapter': chapter, 'standard_code': code},
                ))
            else:
                for j, sub in enumerate(splitter.split_text(content)):
                    if len(sub.strip()) < 100:
                        continue
                    docs.append(Document(
                        page_content=sub.strip(),
                        metadata={
                            'chapter':       chapter,
                            'standard_code': code,
                            'chunk_index':   j,
                        },
                    ))

        chapter_chunks[chapter] = docs
        unique = len(set(d.metadata['standard_code'] for d in docs))
        print(f'  {chapter}: {len(docs)} chunks across {unique} standards')

    return chapter_chunks


pages          = load_pdf(JCI_PDF)
chapter_pages  = group_pages_by_chapter(pages)
chapter_chunks = build_chapter_chunks(chapter_pages)

# ── Build and save chapter-level FAISS DBs ───────────────────
model_dir = os.path.join(DB_SAVE_DIR, EMBEDDING_MODEL_NAME)

if os.path.exists(model_dir) and any(
    os.path.isdir(os.path.join(model_dir, ch)) for ch in VALID_CODES
):
    print(f'[{EMBEDDING_MODEL_NAME}] JCI DBs already exist — skipping build.')
else:
    print(f'[{EMBEDDING_MODEL_NAME}] Building JCI chapter DBs...')
    os.makedirs(model_dir, exist_ok=True)
    jci_build_embeddings = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL_STRING,
        encode_kwargs={'normalize_embeddings': True},
    )
    for chapter, docs in chapter_chunks.items():
        if docs:
            db = FAISS.from_documents(docs, jci_build_embeddings)
            db.save_local(os.path.join(model_dir, chapter))
            print(f'  Saved: {chapter}')
    print(f'[{EMBEDDING_MODEL_NAME}] JCI DBs done.')

print('Section 3 complete.')

Loaded 406 pages from JCI PDF.
HRP: removed 13 end-of-book pages.

=== Chapter page counts ===
  UNKNOWN: 2 pages
  FMS: 42 pages
  APR: 18 pages
  ACC: 22 pages
  PCC: 21 pages
  AOP: 30 pages
  COP: 32 pages
  ASC: 16 pages
  QPS: 15 pages
  PCI: 27 pages
  GLD: 32 pages
  SQE: 28 pages
  MOI: 25 pages
  MPE: 13 pages
  HRP: 32 pages
  IPSG: 16 pages
  MMU: 22 pages
  UNKNOWN: 1 chunks across 1 standards
  FMS: 131 chunks across 27 standards
  APR: 32 chunks across 12 standards
  ACC: 80 chunks across 19 standards
  PCC: 79 chunks across 22 standards
  AOP: 114 chunks across 36 standards
  COP: 129 chunks across 29 standards
  ASC: 50 chunks across 16 standards
  QPS: 56 chunks across 12 standards
  PCI: 106 chunks across 22 standards
  GLD: 142 chunks across 34 standards
  SQE: 128 chunks across 26 standards
  MOI: 95 chunks across 17 standards
  MPE: 23 chunks across 8 standards
  HRP: 140 chunks across 10 standards
  IPSG: 76 chunks across 14 standards
  MMU: 85 chunks across 19 s

---
## Section 4 — Load BGE-large Embeddings & Retrieval Function

- Loads the BGE-large embedding model (shared across all DBs)
- Loads all JCI chapter FAISS DBs and the policy FAISS DB into memory
- Defines `retrieve()` — queries one or more DBs, deduplicates, returns top-`FINAL_K` chunks by L2 distance

In [4]:
print(f'Loading embeddings model: {EMBEDDING_MODEL_STRING}')

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_STRING,
    encode_kwargs={'normalize_embeddings': True},
)

# ── Load JCI chapter DBs ─────────────────────────────────────
jci_dbs = {}
for chapter in chapter_chunks.keys():
    chapter_path = os.path.join(DB_SAVE_DIR, EMBEDDING_MODEL_NAME, chapter)
    if os.path.exists(chapter_path):
        jci_dbs[chapter] = FAISS.load_local(
            chapter_path, embeddings,
            allow_dangerous_deserialization=True,
        )
print(f'Loaded {len(jci_dbs)} JCI chapter DBs.')

# ── Load policy DB ───────────────────────────────────────────
policy_db = FAISS.load_local(
    policy_db_path, embeddings,
    allow_dangerous_deserialization=True,
)
print('Policy DB loaded.')


def retrieve(query: str, dbs: dict) -> list:
    """
    Queries every DB in dbs, sorts globally by FAISS L2 distance
    (lower = more similar), deduplicates, returns top FINAL_K chunks.
    Returns: list of (Document, raw_l2_score)
    """
    pool = []
    for chapter, db in dbs.items():
        for doc, score in db.similarity_search_with_score(query, k=PER_DB_K):
            doc.metadata['chapter'] = chapter
            pool.append((doc, score))

    pool.sort(key=lambda x: x[1])  # ascending: lowest L2 = most similar

    seen, unique = set(), []
    for doc, score in pool:
        key = (doc.page_content, doc.metadata.get('standard_code'))
        if key not in seen:
            unique.append((doc, score))
            seen.add(key)

    return unique[:FINAL_K]


print('Section 4 complete — embeddings and retrieval ready.')

Loading embeddings model: BAAI/bge-large-en-v1.5


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 6227.92it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded 17 JCI chapter DBs.
Policy DB loaded.
Section 4 complete — embeddings and retrieval ready.


---
## Section 5 — Load Qwen Reranker

- Loads `Qwen3-Reranker-4B` as a causal LM
- Scores each (query, chunk) pair as a yes/no relevance judgement
- `norm_score` = log-softmax yes-probability → absolute 0–1
- Score near **1.0** = model is near-certain the chunk is relevant
- Updated instruction prompt is compliance-aware rather than generic QA

In [5]:
print(f'Loading Qwen reranker: {QWEN_MODEL_ID}')

qwen_r_tok = AutoTokenizer.from_pretrained(QWEN_MODEL_ID, padding_side='left')
qwen_r_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL_ID, device_map='auto'
).eval()

qwen_r_true_id  = qwen_r_tok.convert_tokens_to_ids('yes')
qwen_r_false_id = qwen_r_tok.convert_tokens_to_ids('no')
qwen_r_maxlen   = 8192

_qwen_prefix = (
    '<|im_start|>system\n'
    'Judge whether the Document meets the requirements based on the '
    'Query and the Instruct provided. '
    'Note that the answer can only be "yes" or "no".'
    '<|im_end|>\n<|im_start|>user\n'
)
_qwen_suffix = (
    '<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'
)
qwen_r_prefix_toks = qwen_r_tok.encode(_qwen_prefix, add_special_tokens=False)
qwen_r_suffix_toks = qwen_r_tok.encode(_qwen_suffix, add_special_tokens=False)

_QWEN_INSTRUCTION = (
    'Given a hospital accreditation policy compliance query, retrieve the '
    'document chunk that contains the relevant JCI standard, intent, or '
    'measurable elements needed to assess compliance.'
)


def rerank_qwen(query: str, retrieved: list) -> list:
    """
    Returns list of (doc, norm_score) sorted by norm_score descending.
    norm_score = log_softmax yes-probability — absolute 0-1.
    """
    docs  = [doc for doc, _ in retrieved]
    pairs = [
        f'<Instruct>: {_QWEN_INSTRUCTION}\n'
        f'<Query>: {query}\n'
        f'<Document>: {doc.page_content[:400]}'
        for doc in docs
    ]

    inputs = qwen_r_tok(
        pairs,
        padding=False,
        truncation='longest_first',
        return_attention_mask=False,
        max_length=qwen_r_maxlen - len(qwen_r_prefix_toks) - len(qwen_r_suffix_toks),
    )
    for i, ele in enumerate(inputs['input_ids']):
        inputs['input_ids'][i] = qwen_r_prefix_toks + ele + qwen_r_suffix_toks
    inputs = qwen_r_tok.pad(
        inputs, padding=True, return_tensors='pt', max_length=qwen_r_maxlen
    )
    for key in inputs:
        inputs[key] = inputs[key].to(qwen_r_model.device)

    with torch.no_grad():
        logits = qwen_r_model(**inputs).logits[:, -1, :]
        tv     = logits[:, qwen_r_true_id]
        fv     = logits[:, qwen_r_false_id]
        scores = torch.nn.functional.log_softmax(
            torch.stack([fv, tv], dim=1), dim=1
        )[:, 1].exp().tolist()

    combined = [(doc, round(score, 4)) for doc, score in zip(docs, scores)]
    return sorted(combined, key=lambda x: x[1], reverse=True)


print(f'Section 5 complete — Qwen reranker loaded.')

Loading Qwen reranker: Qwen/Qwen3-Reranker-4B


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 417.64it/s]


Section 5 complete — Qwen reranker loaded.


---
## Section 6 — Query Parser

Rule-based parser — **no LLM call**. Extracts four fields from any query:

| Field | What it extracts | Example |
|-------|------------------|---------|
| `task_type` | What the user wants to do | `compliance_check` |
| `target_policy` | Which hospital policy is referenced | `Transfer Policy` |
| `policy_clause` | Clause number label (not used for routing) | `5.1.1` |
| `reference_chapter` | JCI chapter to route retrieval to | `ACC` |

**Routing priority:**
1. Explicit JCI standard code found (e.g. `ACC.5.1`) → `scope: targeted`
2. JCI chapter mentioned (e.g. `ACC`) → `scope: chapter`
3. Nothing found → `scope: global` (searches all DBs)

A **sanity check** runs automatically at the end of this cell.

In [2]:
@dataclass
class ParsedQuery:
    task_type:          str
    target_policy:      str | None
    policy_clause:      str | None   # label only — not used for routing
    reference_standard: str | None
    reference_chapter:  str | None
    measurable_element:  str | None
    retrieval_scope:    str          # 'targeted' / 'chapter' / 'global'
    routing_source:     str          # 'explicit' / 'inferred' / 'global'
    
    def to_json(self) -> str:
        return json.dumps(asdict(self), indent=2)


def extract_jci_standard(text: str) -> tuple[str | None, str | None]:
    """Extracts JCI standard code e.g. ACC.5.1 and its chapter."""
    pattern = r'\b([A-Z]{2,4})\.(\d+(?:\.\d+){0,2})\b'
    for chapter, sub in re.findall(pattern, text):
        if chapter in VALID_CHAPTERS:
            return f'{chapter}.{sub}', chapter
    for chapter in VALID_CHAPTERS:
        if re.search(rf'\b{chapter}\b', text):
            return None, chapter
    return None, None


def extract_target_policy(text: str) -> str | None:
    """Identifies which hospital policy document is referenced."""
    text_lower = text.lower()
    for key, name in KNOWN_POLICIES.items():
        if key in text_lower:
            return name
    return None


def extract_policy_clause(text: str) -> str | None:
    """Extracts clause number label e.g. 5.1.1. Label only — not used for routing."""
    matches = re.findall(r'\b(\d+\.\d+(?:\.\d+)?)\b', text)
    return matches[0] if matches else None

def extract_measurable_element(text:str) -> str | None:
    """
    Extracts JCI measurable element reference if present
    Handles all formats and normalises to ME x
    """
    match = re.search(r'\bmeasurable\s+elements?\s*(\d+)\b', text, re.IGNORECASE)
    if match:
        return f'ME {match.group(1)}'
    
    match = re.search(r'\bme\s*(\d+)\b', text, re.IGNORECASE)
    if match:
        return f'ME {match.group(1)}'
    
    return None
    

def detect_task_type(text: str) -> str:
    """
    Determines task type by keyword matching.
    Priority: compliance_check > comparison > clarification > general_qa.
    """
    text_lower = text.lower()
    for task_type, keywords in TASK_KEYWORDS.items():
        if any(kw in text_lower for kw in keywords):
            return task_type
    return 'general_qa'


def parse_query(query: str, print_json: bool = True) -> ParsedQuery:
    """
    Master parser — rule-based, no LLM call.
    Routing priority:
      1. Explicit JCI standard code in query  → scope: targeted
      2. JCI chapter mentioned in query       → scope: chapter
      3. Nothing found                        → scope: global (fallback)
    """
    task_type     = detect_task_type(query)
    target_policy = extract_target_policy(query)
    policy_clause = extract_policy_clause(query)
    measurable_element = extract_measurable_element(query)
    std_code, chapter = extract_jci_standard(query)

    if std_code:
        result = ParsedQuery(
            task_type          = task_type,
            target_policy      = target_policy,
            policy_clause      = policy_clause,
            reference_standard = std_code,
            reference_chapter  = chapter,
            measurable_element = measurable_element,
            retrieval_scope    = 'targeted',
            routing_source     = 'explicit',
        )
    elif chapter:
        result = ParsedQuery(
            task_type          = task_type,
            target_policy      = target_policy,
            policy_clause      = policy_clause,
            reference_standard = None,
            reference_chapter  = chapter,
            measurable_element = measurable_element,
            retrieval_scope    = 'chapter',
            routing_source     = 'inferred',
        )
    else:
        result = ParsedQuery(
        task_type          = task_type,
        target_policy      = target_policy,
        policy_clause      = policy_clause,
        reference_standard = None,
        reference_chapter  = None,
        measurable_element = measurable_element,
        retrieval_scope    = 'global',
        routing_source     = 'global',
    )

    if print_json:
        print(result.to_json())
    
    return result


def route_and_retrieve(query: str, policy_text: str,
                        parsed: ParsedQuery, dbs: dict) -> list:
    """
    Builds retrieval query from policy text + question.
    Routes to specific chapter DB(s) or global based on parser output.
    policy_text is semantically richer than the question alone.
    """
    retrieval_query = f"{policy_text} {query}" if policy_text else query

    if parsed.retrieval_scope in ('targeted', 'chapter'):
        if parsed.reference_chapter and parsed.reference_chapter in dbs:
            target_dbs = {parsed.reference_chapter: dbs[parsed.reference_chapter]}
            return retrieve(retrieval_query, target_dbs)

    # Global fallback — search all JCI chapter DBs
    return retrieve(retrieval_query, dbs)


# ── Sanity check ─────────────────────────────────────────────
# Expected:
#   Q1 → task=compliance_check | chapter=ACC | scope=targeted | source=explicit
#   Q2 → task=comparison       | chapter=ACC | scope=chapter  | source=inferred
#   Q3 → task=clarification    | chapter=None | scope=global  | source=global

print('Parser sanity check:')
print('-' * 70)
_test_queries = [
    "Does clause 5.1 of our Transfer Policy comply with ACC.5.1?",
    "Compare our transfer process against ACC requirements",
    "What are the handover requirements for inter-hospital transfer?",
    "Does our policy meet Measurable Element 2 of ACC.5.1?",
    "Is this aligned with ACC.5.1 me 3?",
]
for q in _test_queries:
    print(f'Query: {q}')
    parse_query(q, print_json=True)   # ← JSON prints automatically
    print()

print('Section 6 complete — parser ready.')

Parser sanity check:
----------------------------------------------------------------------
Query: Does clause 5.1 of our Transfer Policy comply with ACC.5.1?
{
  "task_type": "compliance_check",
  "target_policy": "Transfer Policy",
  "policy_clause": "5.1",
  "reference_standard": "ACC.5.1",
  "reference_chapter": "ACC",
  "measurable_element": null,
  "retrieval_scope": "targeted",
  "routing_source": "explicit"
}

Query: Compare our transfer process against ACC requirements
{
  "task_type": "comparison",
  "target_policy": "Transfer Policy",
  "policy_clause": null,
  "reference_standard": null,
  "reference_chapter": "ACC",
  "measurable_element": null,
  "retrieval_scope": "chapter",
  "routing_source": "inferred"
}

Query: What are the handover requirements for inter-hospital transfer?
{
  "task_type": "clarification",
  "target_policy": "Transfer Policy",
  "policy_clause": null,
  "reference_standard": null,
  "reference_chapter": null,
  "measurable_element": null,
  "retriev

---
## Section 7 — Gold Standard Queries

Two query sets:

- **`gold_standard_qa`** — original 25 generic QA queries, kept as baseline reference
- **`policy_gold_standard`** — clinician-vetted policy compliance queries

> ⚠️ **Replace the placeholder entries in `policy_gold_standard`** with your actual clinician-vetted Q&As before running Section 8.

Each policy entry needs:
```python
{
    'query':             str,   # the compliance question
    'policy_text':       str,   # the hospital policy excerpt being checked
    'expected_standard': str,   # e.g. 'ACC.5.1'
    'expected_chapter':  str,   # e.g. 'ACC'
    'task_type':         str,   # 'compliance_check' / 'comparison' / 'clarification'
}
```

In [4]:
# ── Clinician-vetted policy compliance queries ───────────────
# !! REPLACE the placeholder entries below with your actual
#    clinician-vetted Q&As before running Section 8.

policy_gold_standard = [
    # ── placeholder — replace with real clinician Q&As ──────
    {
        'query':              'The current Policy Purpose Statement (1.1) reads: "To provide guidelines for healthcare team to comply during intra-hospital and inter-hospital transfers of patients." Does this purpose statement comply with JCI standards for continuity of care, and how does it compare to what JCI ACC.3 requires in terms of scope and integration across care settings?',
        'policy_text':        (
            'To provide guidelines for healthcare team to comply during intra-hospital and inter-hospital transfers of patients'
        ),
        'expected_standard':  'ACC.3',
        'expected_chapter':   'ACC',
        'task_type':          'compliance_check',
    },
    {
        'query':              'Section 3.7 on inter-hospital transfer communication states: "communication of the details of the transfer and the condition of the patient shall be done between doctors of at least Associate Consultant grade and above from both the referring and accepting team." Does this section meet the requirements of JCI ACC.5.1, and what gaps exist between this communication requirement and what the standard mandates in terms of content and documentation?',
        'policy_text':        (
            'communication of the details of the transfer and the condition of the patient shall be done between doctors of at least Associate Consultant grade and above from both the referring and accepting team.'
        ),
        'expected_standard':  'ACC.5.1',
        'expected_chapter':   'ACC',
        'task_type':          'compliance_check',
    },
    {
        'query':              'Section 5.1.4 requires the referring team to "summarize the case history of patient and document it in SCM before transferring the patient to the other discipline," including health status, summary of care, patient progress, and reason for transfer. How does this documentation requirement compare to what JCI ACC.3 Measurable Element 5 specifies, and are there any elements missing that would constitute a compliance gap?',
        'policy_text':        (
            ' '
        ),
        'expected_standard':  'ACC.3',
        'expected_chapter':   'ACC',
        'task_type':          'compliance_check',
    },
    {
        'query':              'Section 5.2.10 states that "critically ill patients requiring inter-hospital transfer should be accompanied by a doctor of at least MO or PGY2 Resident grade" who assumes responsibility for en-route medical control. Does this section fully satisfy JCI ACC.5 Measurable Element 3, and how does its scope of coverage compare to what the standard requires across different patient types?',
        'policy_text':        (
            'critically ill patients requiring inter-hospital transfer should be accompanied by a doctor of at least MO or PGY2 Resident grade'
        ),
        'expected_standard':  'ACC.5',
        'expected_chapter':   'ACC',
        'task_type':          'compliance_check',
    },
    {
        'query':              'Section 5.2.11 states that "accompanying personnel shall be adequately equipped for any interventions required or anticipated," citing endotracheal intubation as an example. How does this approach to equipment planning compare to what JCI ACC.5 Measurable Element 4 requires, and does the current policy language meet the standards expectations for systematic documentation?',
        'policy_text':        (
            'accompanying personnel shall be adequately equipped for any interventions required or anticipated'
        ),
        'expected_standard':  'ACC.5',
        'expected_chapter':   'ACC',
        'task_type':          'compliance_check',
    },
    {
        'query':              'Section 5.2.12 places responsibility on the accepting doctor to assess "that the patient requires transfer and is suitable for transfer before agreeing to transfer the patient." Does this section align with JCI ACC.5 intent regarding the basis for inter-hospital transfers, and what does the standard suggest should be defined at the policy level beyond the accepting doctors individual assessment?',
        'policy_text':        (
            'that the patient requires transfer and is suitable for transfer before agreeing to transfer the patient.'
        ),
        'expected_standard':  'ACC.5',
        'expected_chapter':   'ACC',
        'task_type':          'compliance_check',
    },
    {
        'query':              'Section 5.2.13 states that "the accepting doctor is also responsible for coordinating with other disciplines for the patients total care." How does this compare to what JCI ACC.3 requires for multidisciplinary coordination, and does assigning responsibility to a single doctor satisfy the standards expectations for documented coordination processes?',
        'policy_text':        (
            'the accepting doctor is also responsible for coordinating with other disciplines for the patients total care'
        ),
        'expected_standard':  'ACC.3',
        'expected_chapter':   'ACC',
        'task_type':          'compliance_check',
    },
    {
        'query':              'Section 5.2.14 outlines the accepting doctors responsibilities for ensuring bed availability and reviewing the patient on arrival, including liaising with ICU or overflow ward doctors. How does this section compare to the requirements under JCI ACC.2 and AOP.2, and what gaps exist between the current policys approach to bed management and the standards expectation for a formalised, hospital-wide patient flow process?',
        'policy_text':        (
            ' '
        ),
        'expected_standard':  'ACC.2, AOP.2',
        'expected_chapter':   'ACC, AOP',
        'task_type':          'compliance_check',
    },
    {
        'query':              'Section 5.2.15 states that "outpatient facilities should not be used as a location for doctors to assess inpatients from other hospitals to determine suitability and need for transfer." Does this exclusion align with JCI ACC.1 and AOP.1 requirements for patient assessment and care setting appropriateness, and what does the standard require in terms of formalising assessment processes across inpatient and outpatient settings?',
        'policy_text':        (
            'outpatient facilities should not be used as a location for doctors to assess inpatients from other hospitals to determine suitability and need for transfer.'
        ),
        'expected_standard':  'ACC.1, AOP.1',
        'expected_chapter':   'ACC, AOP.1',
        'task_type':          'compliance_check',
    },
    # ── add more entries here ────────────────────────────────
]

print(f'Policy gold standard: {len(policy_gold_standard)} queries loaded.')
print('Section 7 complete.')

Policy gold standard: 9 queries loaded.
Section 7 complete.


In [6]:
for i, item in enumerate(policy_gold_standard, 1):
    query        = item['query']
    policy_text  = item['policy_text']
    expected_std = item['expected_standard']
    expected_ch  = item['expected_chapter']

    print(f'[{i}/{len(policy_gold_standard)}] Query: {query} \n Expected: {expected_std}')

    # 1. Parse
    parsed = parse_query(query, print_json=True)

[1/9] Query: The current Policy Purpose Statement (1.1) reads: "To provide guidelines for healthcare team to comply during intra-hospital and inter-hospital transfers of patients." Does this purpose statement comply with JCI standards for continuity of care, and how does it compare to what JCI ACC.3 requires in terms of scope and integration across care settings? 
 Expected: ACC.3
{
  "task_type": "compliance_check",
  "target_policy": "Transfer Policy",
  "policy_clause": "1.1",
  "reference_standard": "ACC.3",
  "reference_chapter": "ACC",
  "measurable_element": null,
  "retrieval_scope": "targeted",
  "routing_source": "explicit"
}
[2/9] Query: Section 3.7 on inter-hospital transfer communication states: "communication of the details of the transfer and the condition of the patient shall be done between doctors of at least Associate Consultant grade and above from both the referring and accepting team." Does this section meet the requirements of JCI ACC.5.1, and what gaps exist bet

---
## Section 8 — Validation Loop (Retrieval Only — No LLM)

For each clinician Q&A:
1. **Parse** the query → routing decision
2. **Route + retrieve** from JCI DBs using policy text + query
3. **Rerank** with Qwen
4. **Record** top-1 and top-3 standard matches vs expected





In [ ]:
validation_rows = []

for i, item in enumerate(policy_gold_standard, 1):
    query        = item['query']
    policy_text  = item['policy_text']
    expected_std = item['expected_standard']
    expected_ch  = item['expected_chapter']

    print(f'[{i:02d}/{len(policy_gold_standard)}] Expected: {expected_std}')

    # 1. Parse
    parsed = parse_query(query, print_json=False)

    # 2. Route + retrieve from JCI DBs
    retrieved = route_and_retrieve(query, policy_text, parsed, jci_dbs)

    # 3. Rerank with Qwen — pass policy_text + query for full context
    rerank_query = f"{policy_text} {query}"
    qwen_ranked  = rerank_qwen(rerank_query, retrieved)

    # 4. Collect results
    qwen_top1_doc,  qwen_top1_score = qwen_ranked[0]
    qwen_top3                       = qwen_ranked[:3]
    qwen_top3_stds                  = [doc.metadata.get('standard_code') for doc, _ in qwen_top3]
    qwen_top3_scores                = [score for _, score in qwen_top3]

    top1_match = qwen_top1_doc.metadata.get('standard_code') == expected_std
    top3_match = expected_std in str(qwen_top3_stds)

    validation_rows.append({
        # Query info
        'query':              query,
        'policy_text':        policy_text[:120] + '...',
        'expected_standard':  expected_std,
        'expected_chapter':   expected_ch,
        'gold_task_type':     item['task_type'],
        # Parser output
        'parsed_task_type':   parsed.task_type,
        'parsed_chapter':     parsed.reference_chapter,
        'parsed_measurable_element': parsed.measurable_element,
        'parsed_standard':    parsed.reference_standard,
        'routing_source':     parsed.routing_source,
        'retrieval_scope':    parsed.retrieval_scope,
        'target_policy':      parsed.target_policy,
        # Qwen retrieval results
        'qwen_top1_standard': qwen_top1_doc.metadata.get('standard_code'),
        'qwen_top1_score':    qwen_top1_score,
        'qwen_top1_chunk':    qwen_top1_doc.page_content.strip()[:300],
        'qwen_top3_standards':str(qwen_top3_stds),
        'qwen_top3_scores':   str(qwen_top3_scores),
        # Accuracy flags
        'top1_match':         top1_match,
        'top3_match':         top3_match,
    })

    status = '\u2713' if top1_match else '\u2717'
    print(
        f'  [{status}] Got: {qwen_top1_doc.metadata.get("standard_code")} '
        f'(score: {qwen_top1_score:.4f}) | '
        f'scope: {parsed.retrieval_scope} | '
        f'source: {parsed.routing_source}'
    )

# ── Save validation results ───────────────────────────────────
val_df = pd.DataFrame(validation_rows)
val_df.to_csv('PARSER_VALIDATION.csv', index=False)
print(f'\nSaved: PARSER_VALIDATION.csv ({len(val_df)} rows)')
print('Section 8 complete.')

[01/9] Expected: ACC.3
  [✗] Got: ACC.5 (score: 0.9961) | scope: targeted | source: explicit
[02/9] Expected: ACC.5.1
  [✓] Got: ACC.5.1 (score: 1.0000) | scope: targeted | source: explicit
[03/9] Expected: ACC.3
  [✗] Got: ACC.5.1 (score: 0.9922) | scope: targeted | source: explicit
[04/9] Expected: ACC.5
  [✓] Got: ACC.5 (score: 0.9844) | scope: targeted | source: explicit
[05/9] Expected: ACC.5
  [✗] Got: ACC.1 (score: 0.0447) | scope: targeted | source: explicit
[06/9] Expected: ACC.5
  [✓] Got: ACC.5 (score: 1.0000) | scope: targeted | source: explicit
[07/9] Expected: ACC.3
  [✗] Got: ACC.3.1 (score: 0.9883) | scope: targeted | source: explicit
[08/9] Expected: ACC.2, AOP.2
  [✗] Got: ACC.2 (score: 1.0000) | scope: targeted | source: explicit
[09/9] Expected: ACC.1, AOP.1
  [✗] Got: INTRO (score: 0.9922) | scope: targeted | source: explicit

Saved: PARSER_VALIDATION.csv (9 rows)
Section 8 complete.


---
## Section 9 — LLM: Compliance Analysis

> ⏸ **Only run this section once Section 8 retrieval accuracy is satisfactory.**

Takes the top Qwen-reranked JCI chunks and generates a structured compliance assessment using Mistral.

**Output format per query:**
1. `COMPLIANCE STATUS` — Compliant / Partially Compliant / Non-Compliant / Insufficient Information
2. `JUSTIFICATION` — what aligns or conflicts
3. `GAPS` — JCI requirements not addressed by the policy
4. `JCI REFERENCE` — standard code(s) from retrieved chunks

Results saved to `COMPLIANCE_ANALYSIS.csv`.

In [ ]:
COMPLIANCE_SYSTEM_PROMPT = """
You are an expert healthcare accreditation assistant specialised in the
Joint Commission International (JCI) 7th Edition Standards (special edition
for Singapore Ministry of Health).

You will be given:
- A hospital policy excerpt (the TARGET being assessed)
- One or more retrieved JCI standard chunks (the REFERENCE)
- A compliance question

YOUR PRIMARY RULE:
Assess compliance using ONLY the information explicitly stated in the
provided JCI chunk text. Do not use external knowledge, assumptions,
or prior familiarity with JCI standards.

OUTPUT FORMAT:
1. COMPLIANCE STATUS: Compliant / Partially Compliant / Non-Compliant / Insufficient Information
2. JUSTIFICATION: What in the policy text aligns or conflicts with the JCI standard
3. GAPS (if any): Specific requirements in the JCI standard not addressed by the policy
4. JCI REFERENCE: The standard code(s) from the retrieved chunks

PROHIBITED BEHAVIORS:
- Do NOT hallucinate standards or requirements not in the retrieved text
- Do NOT cite clauses or editions unless they appear in the chunk text
- Do NOT provide clinical or legal advice beyond what is stated

If the retrieved chunks do not contain sufficient information to assess
compliance, state: \"The provided JCI ESS excerpts do not contain sufficient
information to assess compliance with this clause.\"
"""


def build_compliance_prompt(query: str, policy_text: str, chunks) -> list:
    """
    Formats policy text + JCI chunks + question into Mistral message list.
    chunks can be a single Document or a list of Documents.
    """
    if isinstance(chunks, Document):
        chunks = [chunks]

    context_parts = []
    for idx, chunk in enumerate(chunks, 1):
        context_parts.append(
            f'[JCI Chunk {idx} | Standard: {chunk.metadata.get("standard_code", "?")} | '
            f'Chapter: {chunk.metadata.get("chapter", "?")}]\n'
            f'{chunk.page_content.strip()}'
        )
    context_str = '\n\n'.join(context_parts)

    user_content = (
        f'HOSPITAL POLICY EXCERPT:\n'
        f'{policy_text}\n\n'
        f'RELEVANT JCI STANDARDS:\n'
        f'{context_str}\n\n'
        f'COMPLIANCE QUESTION:\n'
        f'{query}\n\n'
        f'Assessment (cite the JCI standard code where relevant):'
    )
    return [
        {'role': 'system', 'content': COMPLIANCE_SYSTEM_PROMPT},
        {'role': 'user',   'content': user_content},
    ]


def generate_mistral_answer(messages: list) -> str:
    """Runs Mistral and returns the assistant reply as plain text."""
    output = mistral_pipeline(messages, max_new_tokens=LLM_MAX_TOKENS)
    return output[0]['generated_text'][-1]['content'].strip()


# ── Load Mistral ─────────────────────────────────────────────
print(f'Loading LLM: {LLM_MODEL_ID}')
mistral_pipeline = transformers.pipeline(
    'text-generation',
    model=LLM_MODEL_ID,
    model_kwargs={'torch_dtype': torch.bfloat16},
    device_map='auto',
)
print(f'LLM loaded: {LLM_MODEL_ID}')

In [ ]:
# ── Compliance generation loop ───────────────────────────────
compliance_rows = []

for i, item in enumerate(policy_gold_standard, 1):
    query        = item['query']
    policy_text  = item['policy_text']
    expected_std = item['expected_standard']

    print(f'[{i:02d}/{len(policy_gold_standard)}] Generating: {expected_std}')

    parsed      = parse_query(query, print_json=False)
    retrieved   = route_and_retrieve(query, policy_text, parsed, jci_dbs)
    rerank_q    = f"{policy_text} {query}"
    qwen_ranked = rerank_qwen(rerank_q, retrieved)

    qwen_top1_doc,   qwen_top1_score = qwen_ranked[0]
    qwen_top3_docs   = [doc for doc, _ in qwen_ranked[:3]]
    qwen_top3_stds   = [doc.metadata.get('standard_code') for doc in qwen_top3_docs]
    qwen_top3_scores = [score for _, score in qwen_ranked[:3]]

    answer_top1 = generate_mistral_answer(
        build_compliance_prompt(query, policy_text, qwen_top1_doc)
    )
    answer_top3 = generate_mistral_answer(
        build_compliance_prompt(query, policy_text, qwen_top3_docs)
    )

    compliance_rows.append({
        'query':               query,
        'policy_text':         policy_text[:120] + '...',
        'expected_standard':   expected_std,
        'qwen_top1_standard':  qwen_top1_doc.metadata.get('standard_code'),
        'qwen_top1_score':     qwen_top1_score,
        'qwen_top1_chunk':     qwen_top1_doc.page_content.strip()[:300],
        'answer_top1':         answer_top1,
        'qwen_top3_standards': str(qwen_top3_stds),
        'qwen_top3_scores':    str(qwen_top3_scores),
        'qwen_top3_chunks':    '\n\n---\n\n'.join(
                                   d.page_content.strip() for d in qwen_top3_docs
                               ),
        'answer_top3':         answer_top3,
    })

compliance_df = pd.DataFrame(compliance_rows)
compliance_df.to_csv('COMPLIANCE_ANALYSIS.csv', index=False)
print(f'Saved: COMPLIANCE_ANALYSIS.csv ({len(compliance_df)} rows)')
print('Section 9 complete.')

---
## Section 10 — Summary

Run after **Section 8** to evaluate retrieval and parser accuracy.
Compliance analysis results (Section 9) are in `COMPLIANCE_ANALYSIS.csv`.

In [13]:
print('\n' + '='*60)
print('PARSER VALIDATION SUMMARY')
print('='*60)
print(f'Total queries evaluated : {len(val_df)}')
print(f'Top-1 accuracy          : {val_df["top1_match"].mean()*100:.1f}%')
print(f'Top-3 accuracy          : {val_df["top3_match"].mean()*100:.1f}%')

print('\nAccuracy by routing source:')
print(
    val_df.groupby('routing_source')[['top1_match', 'top3_match']]
    .mean().round(3).to_string()
)

print('\nAccuracy by retrieval scope:')
print(
    val_df.groupby('retrieval_scope')[['top1_match', 'top3_match']]
    .mean().round(3).to_string()
)

print('\nAccuracy by task type:')
print(
    val_df.groupby('parsed_task_type')[['top1_match', 'top3_match']]
    .mean().round(3).to_string()
)

print('\nFailed cases:')
failed = val_df[~val_df['top1_match']]
if len(failed) == 0:
    print('  None \u2014 all top-1 correct!')
else:
    for _, row in failed.iterrows():
        print(
            f"  Expected : {row['expected_standard']}"
            f" | Got    : {row['qwen_top1_standard']}"
            f" | Score  : {row['qwen_top1_score']}"
            f" | Scope  : {row['retrieval_scope']}"
            f" | Source : {row['routing_source']}"
        )

print('\nQwen Score Summary:')
print(f"  Top-1 avg score : {val_df['qwen_top1_score'].mean():.4f}")
print(
    f"  Top-3 avg score : "
    f"{val_df['qwen_top3_scores'].apply(lambda s: np.mean(eval(s))).mean():.4f}"
)
print()
print('If Qwen gives high scores but low accuracy it is confidently wrong.')
print('='*60)


PARSER VALIDATION SUMMARY
Total queries evaluated : 9
Top-1 accuracy          : 33.3%
Top-3 accuracy          : 55.6%

Accuracy by routing source:
                top1_match  top3_match
routing_source                        
explicit             0.333       0.556

Accuracy by retrieval scope:
                 top1_match  top3_match
retrieval_scope                        
targeted              0.333       0.556

Accuracy by task type:
                  top1_match  top3_match
parsed_task_type                        
compliance_check       0.333       0.556

Failed cases:
  Expected : ACC.3 | Got    : ACC.5 | Score  : 0.9961 | Scope  : targeted | Source : explicit
  Expected : ACC.3 | Got    : ACC.5.1 | Score  : 0.9922 | Scope  : targeted | Source : explicit
  Expected : ACC.5 | Got    : ACC.1 | Score  : 0.0447 | Scope  : targeted | Source : explicit
  Expected : ACC.3 | Got    : ACC.3.1 | Score  : 0.9883 | Scope  : targeted | Source : explicit
  Expected : ACC.2, AOP.2 | Got    : ACC.2 